In [5]:
import math
from random import randint


class Pokemon:
    """
    A base class for a generic pokemon
    
    Attributes (Public)
    -------------------
    name : str
        The name of the pokemon
    level : int
        The level of the pokemon that is used to determine its stats
    status : str
        The status of the pokemon as fainted or ready, indicating if it can battle
    current_health : int
        The current health of the pokemon throughout the battle
    """
    
    def __init__(self, name, primary_type, secondary_type, 
                 attack, defense, sp_attack, sp_defense, health, speed, level=1):
        """
        Establishes the relevant information for the pokemon what will be used when making calculations
        and displaying output
        
        Parameters
        ----------
        name : str
            The name of the pokemon
        primary_type : str
            The primary typing of the pokemon that will determine its type matchups
        secondary_type : str
            The secondary typing of the pokemon that will determine its type matchups
        attack : int
            The base attack stat of the pokemon as an integer
        defense : int
            The base defense stat of the pokemon as an integer
        sp_attack : int
            The base special attack stat of the pokemon as an integer
        sp_defense : int
            The base special defense stat of the pokemon as an integer
        health : int
            The base health stat of the pokemon as an integer
        speed : int
            The base speed stat of the pokemon as an integer
        level : int
            The level of the pokemon as an integer
        """
        self.name = name
        # Base stats and type were made private because they shouldn't be altered outside of the class
        self.__type = (primary_type, secondary_type)
        self.level = level
        self.__base_attack = attack
        self.__base_defense = defense
        self.__base_special_attack = sp_attack
        self.__base_special_defense = sp_defense
        self.__base_health = health
        self.__base_speed = speed
        self.status = "Ready"
        # The following attributes are protected but not private so that they can be accessed easily by the
        # superclasses to be added onto or overridden while still showing that they should only be used 
        # internally
        self._moves = {
            "placeholder1": {"power": 50, "type": primary_type, "attack_type": "Physical", "accuracy": 100},
            "placeholder2": {"power": 50, "type": primary_type, "attack_type": "Special", "accuracy": 100},
            "placeholder3": {"target": "attack", "factor": 1.5, "type": primary_type, "attack type": "Status", "accuracy": 100}
        }
        self.__calculate_stats()
        self._weaknesses = []
        self._resistances = []
        self._double_weaknesses = []
        self._double_resistances = []
        self._immunities = []
        
    def __calculate_stats(self):
        """
        Calculates the effective stats of the pokemon using its base stats and level
        """
        # Stats were made private so they couldn't be altered mid-battle while current health was made
        # public so it could update as the battle progressed
        self.__attack = math.floor(((2 * self.__base_attack * self.level) / 100) + 10)
        self.__defense = math.floor(((2 * self.__base_defense * self.level) / 100) + 10)
        self.__special_attack = math.floor(((2 * self.__base_special_attack * self.level) / 100) + 10)
        self.__special_defense = math.floor(((2 * self.__base_special_defense * self.level) / 100) + 10)
        self.__health = math.floor(self.__base_health + 2 * self.level)
        self.__speed = math.floor(self.__base_speed + 2 * self.level)
        self.current_health = self.__health

    def produce_attack(self, other_pokemon, attack_name):
        """
        Generic attack method for all pokemon
        
        Parameters
        ----------
        other_pokemon : Instance of a Pokemon subclass
            The pokemon receiving the move
        attack_name : str
            The name of the attack being used that will determine the move's attributes for damage
            calculations and output
        """
        attack_name = attack_name.lower()
        if self.status == "Fainted":
            print("Pokemon is fainted and can't attack")
        else:
            # Setting attack attributes
            move = self._moves[attack_name]
            attack_type = move["type"]
            attack_attack_type = move["attack type"]
            accuracy = move["accuracy"]
            hit = randint(1, 100)
            if hit > accuracy:
                print("********************")
                print(f"Choose an attack {self.name}:")
                print("The attack missed.")
            else:
                # Conditionals determine how damage will be calculated and what the outputted text will be
                # based on the chosen attack's attributes
                if attack_attack_type == "Physical" or attack_attack_type == "Special":
                    power = move["power"]
                    if attack_attack_type == "Physical":
                        attack = self.get_stat("attack")
                        other_defense = other_pokemon.get_stat("defense")
                    elif attack_attack_type == "Special":
                        attack = self.get_stat("special attack")
                        other_defense = other_pokemon.get_stat("special defense")
                    damage = math.floor(((0.5 * (attack / other_defense) * power) + 1))
                    if attack_type in self.get_type():
                        damage += math.floor(0.2 * damage)
                    other_pokemon.take_damage(damage, attack_type)
                    print("********************")
                    print(f"Choose an attack {self.name}:")
                    print(f"{self.name} uses {attack_name} on {other_pokemon.name}!")
                    if attack_type in self.get_type():
                        print(f"{attack_name} does boosted damage.")
                    else:
                        print(f"{attack_name} does regular damage.")
                    if attack_type in other_pokemon.get_weaknesses() or attack_type in other_pokemon.get_double_weaknesses():
                        print("It's super effective!")
                    elif attack_type in other_pokemon.get_resistances() or attack_type in other_pokemon.get_double_resistances():
                        print("It's not very effective.")
                    elif attack_type in other_pokemon.get_immunities():
                        print(f"It doesn't affect {other_pokemon.name}.")
                    else:
                        print("It's effective")
                    if attack_type in other_pokemon.get_immunities():
                        print(f"{other_pokemon.name} took no damage.")
                    else:
                        print(f"{other_pokemon.name} took {damage} damage. Health is now {other_pokemon.current_health}.")
                    if other_pokemon.status == "Fainted":
                        print(f"{other_pokemon.name} has fainted.")
                    print("********************")
                    # Creates the basic logic for stat lowering status moves
                elif attack_attack_type == "Status":
                    target_stat = move["target"]
                    factor = move["factor"]
                    if factor < 1:
                        new_value = other_pokemon.get_stat(target_stat) * factor
                        other_pokemon.set_stat(target_stat, new_value)
                        print(f"Choose an attack {self.name}:")
                        print(f"{self.name} uses {attack_name} on {other_pokemon.name}!")
                        print(f"{other_pokemon.name}'s {target_stat} was lowered.")
                        print("********************")
                    elif factor > 1:
                        new_value = self.get_stat(target_stat) * factor
                        self.set_stat(target_stat, new_value)
                        print(f"Choose an attack {self.name}:")
                        print(f"{self.name} uses {attack_name} on {other_pokemon.name}!")
                        print(f"{self.name} raised its {target_stat}.")
                        print("********************")

    def take_damage(self, damage, attack_type):
        """
        Generic method for taking damage for all pokemon
        
        Parameters
        ----------
        damage : int
            The damage being inflicted as an integer
        attack_type : str
            The type of the move being used that will determine its effectiveness and damage multiplier
        """
        # Conditionals determine damage multiplier based on the type matchup of the pokemon and move
        if attack_type in self.get_weaknesses():
            damage *= 2
        elif attack_type in self.get_double_weaknesses():
            damage *= 4
        elif attack_type in self.get_resistances():
            damage *= 0.5
        elif attack_type in self.get_double_resistances():
            damage *= 0.25
        elif attack_type in self.get_immunities():
            damage = 0
        self.current_health -= damage
        if self.current_health <= 0:
            self.status = "Fainted"
            self.current_health = 0

    def show_health(self):
        """
        Prints the current health stat of the pokemon
        """
        print("********************")
        print(f"Name: {self.name}")
        print(f"Health: {self.current_health}")
        print("********************")
    
    def get_stat(self, stat):
        """
        Finds the desired private stat of the pokemon
        
        Parameters
        ----------
        stat : str
            The stat being acquired by the method
            
        Returns
        -------
        __attack : int
            The attack stat that is returned if attack was entered as the desired stat
        __defense : int
            The defense stat that is returned if defense was entered as the desired stat
        __special_attack : int
            The special attack stat that is returned if special attack was entered as the desired stat
        __special_defense : int
            The special defense stat that is returned if special defense was entered as the desired stat
        __speed : int
            The speed stat that is returned if speed was entered as the desired stat
        """
        # Allows stats to be displayed while still being private
        stat = stat.lower()
        if stat == "attack":
            return self.__attack
        elif stat == "defense":
            return self.__defense
        elif stat == "special attack":
            return self.__special_attack
        elif stat == "special defense":
            return self.__special_defense
        elif stat == "speed":
            return self.__speed
        else:
            print("Not valid stat")
            
    def set_stat(self, stat, value):
        """
        Changes the value of the desired stat to the new value inputted
        
        Paramters
        ---------
        stat : str
            The stat that will be changed
        value : int
            The new numerical value being given to the stat
        """
        # Allows stats to be changes while still being private
        if value <= 0:
            print("Impossible value")
        else:
            if stat == "attack":
                self.__attack = value
            elif stat == "defense":
                self.__defense = value
            elif stat == "special attack":
                self.__special_attack = value
            elif stat == "special defense":
                self.__special_defense = value
            elif stat == "speed":
                self.__speed = value
            else:
                print("Not valid stat")
                
    def get_type(self):
        """
        Finds the type of the pokemon
        
        Returns
        -------
        __type : tuple
            The type of the pokemon
        """
        return self.__type
    
    # Get functions are created despite the attributes being protected and not private because 
    # they should only be changed in the class itself
    
    def get_weaknesses(self):
        """
        Finds the weaknesses of the pokemon
        
        Returns
        -------
        _weaknesses : list
            The weaknesses of the pokemon
        """
        return self._weaknesses
    
    def get_resistances(self):
        """
        Finds the resistances of the pokemon
        
        Returns
        -------
        _resistances : list
            The resistances of the pokemon
        """
        return self._resistances
    
    def get_double_weaknesses(self):
        """
        Finds the double weaknesses of the pokemon
        
        Returns
        -------
        _double_weaknesses : list
            The double weaknesses of the pokemon
        """
        return self._double_weaknesses
    
    def get_double_resistances(self):
        """
        Finds the double resistances of the pokemon
        
        Returns
        -------
        _double_resistances : list
            The double resistances of the pokemon
        """
        return self._double_resistances
    
    def get_immunities(self):
        """
        Finds the immunities of the pokemon
        
        Returns
        -------
        _immunities : list
            The immunities of the pokemon
        """
        return self._immunities

# All the pokemon's base stats, moves, typings, and type matchups were taken from the games themselves

class Pikachu(Pokemon):
    """
    A subclass of Pokemon for the Pikachu species
    
    Attributes (Public)
    -------------------
    name : str
        The name of the pokemon, Pikachu
    level : int
        The level of the Pikachu that is used to determine its stats
    status : str
        The status of the Pikachu as fainted or ready, indicating if it can battle
    current_health : int
        The current health of the Pikachu throughout the battle
    """
    
    def __init__(self, level):
        """
        Plugs in Pikachu's information into the base pokemon initialization method, establishes its type
        matchups, and overrides the placeholder moves with a Pikachu specific moveset
        
        Parameters
        ----------
        level : int
            The level of the Pikachu as an integer
        """
        Pokemon.__init__(self, "Pikachu", "Electric", "None",
                         55, 40, 50, 50, 35, 90, level)
        self._moves = {
            "surf": {"power": 60, "type": "Water", "attack type": "Special", "accuracy": 100}, 
            "thunderbolt": {"power": 60, "type": "Electric", "attack type": "Special", "accuracy": 100},
            "iron tail": {"power": 100, "type": "Steel", "attack type": "Physical", "accuracy": 75},
            "agility": {"target": "speed", "factor": 2, "type": "Psychic", "attack type": "Status", "accuracy": 100}
        }
        self._weaknesses.append("Ground")
        self._resistances.extend(["Electric", "Flying", "Steel"])
        
    def produce_attack(self, other_pokemon, attack_name):
        """
        Uses the base pokemon produce attack function and adds extra conditionals to accounts for the
        potential stat changes from the move iron tail
        
        Parameters
        ----------
         other_pokemon : Instance of a Pokemon subclass
            The pokemon receiving the move
        attack_name : str
            The name of the attack being used that will determine the move's attributes for damage
            calculations, outputs, and additional effects
        """
        Pokemon.produce_attack(self, other_pokemon, attack_name)
        # Accounts for iron tail's 30% chance of lowering defense by one stage
        if attack_name.lower() == "iron tail" and other_pokemon.status != "Fainted":
            chance = randint(1, 100)
            if chance > 70:
                new_defense = other_pokemon.get_stat("defense") * 0.67
                other_pokemon.set_stat("defense", new_defense)
                print(f"{other_pokemon.name}'s defense has been lowered.'")
                print("********************")

class Axew(Pokemon):
    """
    A subclass of Pokemon for the Axew species
    
    Attributes (Public)
    -------------------
    name : str
        The name of the pokemon, Axew
    level : int
        The level of the Axew that is used to determine its stats
    status : str
        The status of the Axew as fainted or ready, indicating if it can battle
    current_health : int
        The current health of the Axew throughout the battle
    """
    
    def __init__(self, level):
        """
        Plugs in Axew's information into the base pokemon initialization method, establishes its type
        matchups, and overrides the placeholder moves with an Axew specific moveset
        
        Parameters
        ----------
        level : int
            The level of the Axew as an integer
        """
        Pokemon.__init__(self, "Axew", "Dragon", "None",
                         87, 60, 30, 40, 46, 57, level)
        self._moves = {
            "dragon claw": {"power": 80, "type": "Dragon", "attack type": "Physical", "accuracy": 100}, 
            "crunch": {"power": 80, "type": "Dark", "attack type": "Physical", "accuracy": 100},
            "stomping tantrum": {"power": 75, "type": "Ground", "attack type": "Physical", "accuracy": 100},
            "leer": {"target": "defense", "factor": 0.67, "type": "Normal", "attack type": "Status", "accuracy": 100}
        }
        self._weaknesses.extend(["Dragon", "Ice", "Fairy"])
        self._resistances.extend(["Electric", "Fire", "Water", "Grass"])
        
    def produce_attack(self, other_pokemon, attack_name):
        """
        Uses the base pokemon produce attack function and adds extra conditionals to accounts for the
        potential stat changes from the move crunch
        
        Parameters
        ----------
         other_pokemon : Instance of a Pokemon subclass
            The pokemon receiving the move
        attack_name : str
            The name of the attack being used that will determine the move's attributes for damage
            calculations, outputs, and additional effects
        """
        Pokemon.produce_attack(self, other_pokemon, attack_name)
        # Accounts for crunch's 20% chance of lowering defense by one stage
        if attack_name.lower() == "crunch" and other_pokemon.status != "Fainted":
            chance = randint(1, 100)
            if chance > 80:
                new_defense = other_pokemon.get_stat("defense") * 0.67
                other_pokemon.set_stat("defense", new_defense)
                print(f"{other_pokemon.name}'s defense has been lowered.'")
                print("********************")
                        
class Psyduck(Pokemon):
    """
    A subclass of Pokemon for the Psyduck species
    
    Attributes (Public)
    -------------------
    name : str
        The name of the pokemon, Psyduck
    level : int
        The level of the Psyduck that is used to determine its stats
    status : str
        The status of the Psyduck as fainted or ready, indicating if it can battle
    current_health : int
        The current health of the Psyduck throughout the battle
    """
    
    def __init__(self, level):
        """
        Plugs in Psyduck's information into the base pokemon initialization method, establishes its type
        matchups, and overrides the placeholder moves with a Psyduck specific moveset
        
        Parameters
        ----------
        level : int
            The level of the Psyduck as an integer
        """
        Pokemon.__init__(self, "Psyduck", "Water", "None",
                         52, 48, 65, 50, 50, 55, level)
        self._moves = {
            "aqua tail": {"power": 90, "type": "Water", "attack type": "Physical", "accuracy": 90}, 
            "blizzard": {"power": 110, "type": "Ice", "attack type": "Special", "accuracy": 70},
            "psychic": {"power": 90, "type": "Psychic", "attack type": "Special", "accuracy": 100},
            "amnesisa": {"target": "special defense", "factor": 2, "type": "Psychic", "attack type": "Status", "accuracy": 100}
        }
        self._weaknesses.extend(["Grass", "Electric"])
        self._resistances.extend(["Water", "Fire", "Steel", "Ice"])
        
    def produce_attack(self, other_pokemon, attack_name):
        """
        Uses the base pokemon produce attack function and adds extra conditionals to accounts for the
        potential stat changes from the move psychic
        
        Parameters
        ----------
         other_pokemon : Instance of a Pokemon subclass
            The pokemon receiving the move
        attack_name : str
            The name of the attack being used that will determine the move's attributes for damage
            calculations, outputs, and additional effects
        """
        Pokemon.produce_attack(self, other_pokemon, attack_name)
        # Accounts for psychic's 10% chance of loweing special defense by one stage
        if attack_name.lower() == "psychic" and other_pokemon.status != "Fainted" and "Psychic" not in other_pokemon.get_immunities():
            chance = randint(1, 100)
            if chance > 90:
                new_special_defense = other_pokemon.get_stat("special defense") * 0.67
                other_pokemon.set_stat("special defense", new_special_defense)
                print(f"{other_pokemon.name}'s special defense has been lowered.'")
                print("********************")

class Cubone(Pokemon):
    """
    A subclass of Pokemon for the Cubone species
    
    Attributes (Public)
    -------------------
    name : str
        The name of the pokemon, Cubone
    level : int
        The level of the Cubone that is used to determine its stats
    status : str
        The status of the Cubone as fainted or ready, indicating if it can battle
    current_health : int
        The current health of the Cubone throughout the battle
    """
    
    def __init__(self, level):
        """
        Plugs in Cubone's information into the base pokemon initialization method, establishes its type
        matchups, and overrides the placeholder moves with a Cubone specific moveset
        
        Parameters
        ----------
        level : int
            The level of the Cubone as an integer
        """
        Pokemon.__init__(self, "Cubone", "Ground", "None",
                         50, 95, 40, 50, 50, 35, level)
        self._moves = {
            "bulldoze": {"power": 60, "type": "Ground", "attack type": "Physical", "accuracy": 100}, 
            "flamethrower": {"power": 90, "type": "Fire", "attack type": "Special", "accuracy": 100},
            "brick break": {"power": 75, "type": "Fighting", "attack type": "Physical", "accuracy": 100},
            "iron defense": {"target": "defense", "factor": 2, "type": "Steel", "attack type": "Status", "accuracy": 100}
        }
        self._weaknesses.extend(["Grass", "Water", "Ice"])
        self._resistances.extend(["Poison", "Rock"])
        self._immunities.append("Electric")
        
    def produce_attack(self, other_pokemon, attack_name):
        """
        Uses the base pokemon produce attack function and adds extra conditionals to accounts for the
        additional stat changes from the move bulldoze
        
        Parameters
        ----------
         other_pokemon : Instance of a Pokemon subclass
            The pokemon receiving the move
        attack_name : str
            The name of the attack being used that will determine the move's attributes for damage
            calculations, outputs, and additional effects
        """
        Pokemon.produce_attack(self, other_pokemon, attack_name)
        # Accounts for bulldoze always lowering speed by one stage
        if attack_name.lower() == "bulldoze" and other_pokemon.status != "Fainted" and "Ground" not in other_pokemon.get_immunities():
            new_speed = other_pokemon.get_stat("speed") * 0.67
            other_pokemon.set_stat("speed", new_speed)
            print(f"{other_pokemon.name}'s speed has been lowered.'")
            print("********************")
                        
class Natu(Pokemon):
    """
    A subclass of Pokemon for the Natu species
    
    Attributes (Public)
    -------------------
    name : str
        The name of the pokemon, Natu
    level : int
        The level of the Natu that is used to determine its stats
    status : str
        The status of the Natu as fainted or ready, indicating if it can battle
    current_health : int
        The current health of the Natu throughout the battle
    """
    
    def __init__(self, level):
        """
        Plugs in Natu's information into the base pokemon initialization method, establishes its type
        matchups, and overrides the placeholder moves with a Natu specific moveset
        
        Parameters
        ----------
        level : int
            The level of the Natu as an integer
        """
        Pokemon.__init__(self, "Natu", "Psychic", "Flying",
                         50, 45, 70, 45, 40, 70, level)
        self._moves = {
            "psychic": {"power": 90, "type": "Psychic", "attack type": "Special", "accuracy": 100}, 
            "aerial ace": {"power": 60, "type": "Flying", "attack type": "Physical", "accuracy": 100},
            "ominous wind": {"power": 60, "type": "Ghost", "attack type": "Special", "accuracy": 100},
            "dazzling gleam": {"power": 80, "type": "Fairy", "attack type": "Special", "accuracy": 100}
        }
        self._weaknesses.extend(["Electric", "Ice", "Rock", "Ghost", "Dark"])
        self._resistances.extend(["Grass", "Psychic"])
        self._double_resistances.append("Fighting")
        self._immunities.append("Ground")
        
    def produce_attack(self, other_pokemon, attack_name):
        """
        Uses the base pokemon produce attack function and adds extra conditionals to accounts for the
        potential stat changes from the moves ominous wind and psychic
        
        Parameters
        ----------
         other_pokemon : Instance of a Pokemon subclass
            The pokemon receiving the move
        attack_name : str
            The name of the attack being used that will determine the move's attributes for damage
            calculations, outputs, and additional effects
        """
        Pokemon.produce_attack(self, other_pokemon, attack_name)
        # Accounts for ominous wind's 10$ chance of raising all of the pokemon's stats by one stage and 
        # psychic's 10% chance of loweing special defense by one stage
        if attack_name.lower() == "ominous wind" and other_pokemon.status != "Fainted" and "Ghost" not in other_pokemon.get_immunities():
            boost = randint(1, 100)
            if boost > 90:
                stats_to_raise = ["attack", "defense", "special attack", "special defense", "speed"]
                for stat in stats_to_raise:
                    new_stat = self.get_stat(stat) * 1.5
                    self.set_stat(stat, new_stat)
                print(f"All of {self.name}'s stats have been raised.")
                print("********************")
        elif attack_name.lower() == "psychic" and other_pokemon.status != "Fainted" and "Psychic" not in other_pokemon.get_immunities():
            chance = randint(1, 100)
            if chance > 90:
                new_special_defense = other_pokemon.get_stat("special defense") * 0.67
                other_pokemon.set_stat("special defense", new_special_defense)
                print(f"{other_pokemon.name}'s special defense has been lowered.'")
                print("********************")
                        
def main():
    """
    Conatins all the logic of the program that allows for the game to be played
    """
    pokemon_roster = ["pikachu", "axew", "psyduck", "cubone", "natu"]
    # While loops are used to ensure that a valid option is inputted before conitnuing, preventing
    # potential crashes
    print(f"Pokemon Roster: {pokemon_roster}")
    while True:
        pokemon = input("Player 1, choose your pokemon! ")
        if pokemon.lower() in pokemon_roster:
            while True:
                level = int(input("What leverl is it? (1-100) "))
                if level > 100 or level < 1:
                    print("Invalid. Level must be a number between 1 and 100.")
                else:
                    break
            break
        else:
            print("Ivalid pokemon. Please select from the roster.")
    if pokemon == "Pikachu" or pokemon == "pikachu":
        pokemon1 = Pikachu(level)
    elif pokemon == "Axew" or pokemon == "axew":
        pokemon1 = Axew(level)
    elif pokemon == "Psyduck" or pokemon == "psyduck":
        pokemon1 = Psyduck(level)
    elif pokemon == "Cubone" or pokemon == "cubone":
        pokemon1 = Cubone(level)
    elif pokemon == "Natu" or pokemon == "natu":
        pokemon1 = Natu(level)
    while True:
        pokemon = input("Player 2, choose your pokemon! ")
        if pokemon.lower() in pokemon_roster:
            while True:
                level = int(input("What leverl is it? (1-100) "))
                if level > 100 or level < 1:
                    print("Invalid. Level must be between 1 and 100.")
                else:
                    break
            break
        else:
            print("Ivalid pokemon. Please select from the roster")
    if pokemon == "Pikachu" or pokemon == "pikachu":
        pokemon2 = Pikachu(level)
    elif pokemon == "Axew" or pokemon == "axew":
        pokemon2 = Axew(level)
    elif pokemon == "Psyduck" or pokemon == "psyduck":
        pokemon2 = Psyduck(level)
    elif pokemon == "Cubone" or pokemon == "cubone":
        pokemon2 = Cubone(level)
    elif pokemon == "Natu" or pokemon == "natu":
        pokemon2 = Natu(level)
    # Main while loop makes the battle continue until a pokemon faints and other while loope ensure
    # valid options are inputted
    # Current health is shown at the end of bother players' turns to update them on their pokemon's status
    while pokemon1.status == "Ready" and pokemon2.status == "Ready":
        if pokemon1.get_stat("speed") > pokemon2.get_stat("speed"):
            print(f"Move Options: {pokemon1._moves.keys()}")
            while True:
                attack1 = input("Player 1, choose your move: ")
                if attack1.lower() in pokemon1._moves.keys():
                    pokemon1.produce_attack(pokemon2, attack1)
                    break
                else:
                    print(f"Invalid. Available moves are {pokemon1._moves.keys()}")
            if pokemon2.status == "Fainted":
                break
            print(f"Move Options: {pokemon2._moves.keys()}")
            while True:
                attack2 = input("Player 2, choose your move: ")
                if attack2.lower() in pokemon2._moves.keys():
                    pokemon2.produce_attack(pokemon1, attack2)
                    break
                else:
                    print(f"Invalid. Available moves are {pokemon1._moves.keys()}")
            if pokemon1.status == "Fainted":
                break
            pokemon1.show_health()
            pokemon2.show_health()
        elif pokemon1.get_stat("speed") < pokemon2.get_stat("speed"):
            print(f"Move Options: {pokemon2._moves.keys()}")
            while True:
                attack2 = input("Player 2, choose your move: ")
                if attack2.lower() in pokemon2._moves.keys():
                    pokemon2.produce_attack(pokemon1, attack2)
                    break
                else:
                    print(f"Invalid. Available moves are {pokemon1._moves.keys()}")
            if pokemon1.status == "Fainted":
                break
            print(f"Move Options: {pokemon1._moves.keys()}")
            while True:
                attack1 = input("Player 1, choose your move: ")
                if attack1.lower() in pokemon1._moves.keys():
                    pokemon1.produce_attack(pokemon2, attack1)
                    break
                else:
                    print(f"Invalid. Available moves are {pokemon1._moves.keys()}")
            if pokemon2.status == "Fainted":
                break
            pokemon1.show_health()
            pokemon2.show_health()
        else:
            turn = randint(1, 2)
            if turn == 1:
                print(f"Move Options: {pokemon1._moves.keys()}")
                while True:
                    attack1 = input("Player 1, choose your move: ")
                    if attack1.lower() in pokemon1._moves.keys():
                        pokemon1.produce_attack(pokemon2, attack1)
                        break
                    else:
                        print(f"Invalid. Available moves are {pokemon1._moves.keys()}")
                if pokemon2.status == "Fainted":
                    break
                print(f"Move Options: {pokemon2._moves.keys()}")
                while True:
                    attack2 = input("Player 2, choose your move: ")
                    if attack2.lower() in pokemon2._moves.keys():
                        pokemon2.produce_attack(pokemon1, attack2)
                        break
                    else:
                        print(f"Invalid. Available moves are {pokemon1._moves.keys()}")
                if pokemon1.status == "Fainted":
                    break
                pokemon1.show_health()
                pokemon2.show_health()
            elif turn == 2:
                print(f"Move Options: {pokemon2._moves.keys()}")
                while True:
                    attack2 = input("Player 2, choose your move: ")
                    if attack2.lower() in pokemon2._moves.keys():
                        pokemon2.produce_attack(pokemon1, attack2)
                        break
                    else:
                        print(f"Invalid. Available moves are {pokemon1._moves.keys()}")
                if pokemon1.status == "Fainted":
                    break
                print(f"Move Options: {pokemon1._moves.keys()}")
                while True:
                    attack1 = input("Player 1, choose your move: ")
                    if attack1.lower() in pokemon1._moves.keys():
                        pokemon1.produce_attack(pokemon2, attack1)
                        break
                    else:
                        print(f"Invalid. Available moves are {pokemon1._moves.keys()}")
                if pokemon2.status == "Fainted":
                    break
                pokemon1.show_health()
                pokemon2.show_health()
            
main()


Pokemon Roster: ['pikachu', 'axew', 'psyduck', 'cubone', 'natu']
Move Options: dict_keys(['surf', 'thunderbolt', 'iron tail', 'agility'])
********************
Choose an attack Pikachu:
Pikachu uses surf on Axew!
surf does regular damage.
It's not very effective.
Axew took 35 damage. Health is now 68.5.
********************
Move Options: dict_keys(['dragon claw', 'crunch', 'stomping tantrum', 'leer'])
********************
Choose an attack Axew:
Axew uses dragon claw on Pikachu!
dragon claw does boosted damage.
It's effective
Pikachu took 81 damage. Health is now 0.
Pikachu has fainted.
********************
